# Optional Project - Colab Part1 (No Drive)

This notebook runs Task1 only: preprocess, tokenizer, and HF publishing.


In [1]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
DATA_CONFIG = "configs/data.yaml"


In [2]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


Cloning into '/content/optionalproject'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 51 (delta 14), reused 50 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 204.63 KiB | 6.20 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/optionalproject
Already on 'run'
Your branch is up to date with 'origin/run'.
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already up to date.
run
6905d99


In [3]:
# 2) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -U datasets huggingface_hub cairosvg tokenizers


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,601 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 http://arc

In [4]:
# 3) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


HF token loaded from Colab key.
has_hf_token: True


In [5]:
# 4) Render sanity check (must be True before preprocess)
from src.data.validate_svg import validate_render
svg = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24"><circle cx="12" cy="12" r="6"/></svg>'
ok, err = validate_render(svg)
print('render_check_ok:', ok)
print('render_check_err:', err)


render_check_ok: True
render_check_err: None


In [6]:
# 5) Preprocess data (and push clean dataset if hf_push.enabled=true)
%cd $REPO_DIR
!python scripts/run_preprocess.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
[1/8] Checking processed cache + manifest...
  No valid manifest found. Regenerating outputs.
[2/8] Loading source datasets from Hugging Face (uses cache_dir for reuse)...
README.md: 1.76kB [00:00, 818kB/s]
data/train-00000-of-00001.parquet: 100% 137M/137M [00:02<00:00, 52.1MB/s]
data/test-00000-of-00001.parquet: 100% 4.59M/4.59M [00:00<00:00, 11.3MB/s]
data/val-00000-of-00001.parquet: 100% 11.1M/11.1M [00:00<00:00, 53.2MB/s]
Generating train split: 100% 80434/80434 [00:00<00:00, 101251.08 examples/s]
Generating test split: 100% 2682/2682 [00:00<00:00, 84119.43 examples/s]
Generating val split: 100% 6254/6254 [00:00<00:00, 96948.56 examples/s]
README.md: 1.76kB [00:00, 4.15MB/s]
data/train-00000-of-00001.parquet: 100% 12.7M/12.7M [00:01<00:00, 12.6MB/s]
data/test-00000-of-00001.parquet: 100% 1.05M/1.05M [00:00<00:00, 2.56MB/s]
data/val-00000-of-00001.parquet: 100% 687k/687k [00:00<00:00, 3.28MB/s]
Generating train split: 1

In [7]:
# 6) Train tokenizer + encode splits (enforces real train token target)
%cd $REPO_DIR
!python scripts/run_tokenizer.py --config {DATA_CONFIG}


/content/optionalproject
[1/4] Preparing tokenizer training text...
[2/4] Training BPE tokenizer...
[00:00:07] Tokenize words                 ██████████████████ 2276072  /  2276072
[00:00:04] Count pairs                    ██████████████████ 2276072  /  2276072
[00:00:23] Compute merges                 ██████████████████ 4046     /     4046
  tokenizer: data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json
[3/4] Encoding train/validation/test splits...
[4/4] Writing tokenization stats...
Done.
Vocab size: 4096
Token totals:
  train: 284601303
  validation: 2903022
  test: 2899636


In [8]:
# 7) Push tokenizer artifacts to HF model repo
%cd $REPO_DIR
!python scripts/push_tokenizer_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:10913: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")
Pushed tokenizer artifacts to: Zala0429/svg-scaling-tokenizer-v1
{"repo_id": "Zala0429/svg-scaling-tokenizer-v1", "repo_type": "model"}


In [9]:
# 8) Push tokenized dataset to HF dataset repo
%cd $REPO_DIR
!python scripts/push_tokenized_dataset_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
Uploading the dataset shards:   0% 0/3 [00:00<?, ? shards/s]
Creating parquet from Arrow format:   0% 0/4 [00:00<?, ?ba/s]
Creating parquet from Arrow format:  25% 1/4 [00:00<00:01,  2.25ba/s]
Creating parquet from Arrow format:  50% 2/4 [00:00<00:00,  2.23ba/s]
Creating parquet from Arrow format:  75% 3/4 [00:01<00:00,  2.27ba/s]
Creating parquet from Arrow format: 100% 4/4 [00:01<00:00,  2.28ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpq6f779mr.parquet    :  11% 41.2M/385M [00:00<?, ?B/s]

Processing Files (0 / 1)      :  11% 41.2M/385M [00:00<00:05, 67.0MB/s,  103MB/s  ]
New Data Upload               :  31% 41.2M/134M [00:00<00:01, 67.0MB/s,  103MB/s  ]

Processing Files (0 / 1)      :  17% 66.7M/385M [00:00<00:03, 86.2MB/s,  111MB/s  ]
New Data Upload               :  50% 66.7M/134M [00:00<00:00, 86.2MB

In [10]:
# 9) Inspect key outputs
import json
from pathlib import Path
root = Path('data/processed/v1-clean-rawsplit')
stats = root / 'stats.json'
tok = root / 'tokenizer' / 'token_stats.json'
print('stats exists:', stats.exists())
print('token_stats exists:', tok.exists())
if stats.exists():
    s = json.loads(stats.read_text(encoding='utf-8'))
    print('cleaned_records:', s.get('cleaned_records'))
    print('train_token_est_total:', s.get('train_token_est_total'))
if tok.exists():
    t = json.loads(tok.read_text(encoding='utf-8'))
    print('vocab_size:', t.get('vocab_size'))
    print('train_total_tokens:', t.get('splits', {}).get('train', {}).get('total_tokens'))


stats exists: True
token_stats exists: True
cleaned_records: 434828
train_token_est_total: 5375026
vocab_size: 4096
train_total_tokens: 284601303
